# VeriFin — Question categories & model performance

All computed from the run artifacts in `results/`. Run All to refresh.

## Setup: categorize the 600 questions and load per-model accuracy

In [ ]:
import json, re
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "results").exists() and (ROOT.parent / "results").exists():
    ROOT = ROOT.parent

data = {}
for line in (ROOT/"data/multicompany_provable/data/provable_calcrequired.jsonl").read_text().splitlines():
    if line.strip():
        r = json.loads(line); data[r["financebench_id"]] = r

def categorize(q):
    q = q.lower()
    table = [("gross profit","Gross profit","Income stmt"),("operating income","Operating income","Income stmt"),
       ("operating expenses","Operating expenses","Income stmt"),("income before income taxes","Income before taxes","Income stmt"),
       ("net income","Net income","Income stmt"),("total current assets","Current assets","Balance sheet"),
       ("total non-current assets","Non-current assets","Balance sheet"),
       ("total current liabilities","Current liabilities","Balance sheet"),
       ("total non-current liabilities","Non-current liab.","Balance sheet"),
       ("total liabilities","Total liabilities","Balance sheet"),("total shareholders","Shareholders equity","Balance sheet"),
       ("net property","Net PP&E","Balance sheet"),
       ("net cash from operating","Cash: operations","Cash flow"),
       ("net cash from investing","Cash: investing","Cash flow"),
       ("net cash from financing","Cash: financing","Cash flow")]
    for k,v,s in table:
        if k in q: return v, s
    return "other","?"

def _nums(t):
    out=[]
    for x in re.findall(r"-?\d[\d,]*\.?\d*", str(t or "")):
        try: out.append(abs(float(x.replace(",",""))))
        except ValueError: pass
    return out
def _correct(cv, gold):
    if cv is None: return False
    t=_nums(gold)
    return any(abs(abs(float(cv))-x)<=max(1.0,x*0.01) for x in t) if t else False

MODELS = {"claude":"results/mc_calc_detsmt_v2_haiku",
          "qwen2.5-7b":"results/mc_calc_qwen2.5-7b",
          "llama3.1-8b":"results/mc_calc_llama3.1-8b"}

recs = {}
for m, d in MODELS.items():
    p = ROOT/d/"results.jsonl"
    if not p.exists(): continue
    for line in p.read_text().splitlines():
        if not line.strip(): continue
        r = json.loads(line); metric, stmt = categorize(data[r["id"]]["question"])
        key = (metric, stmt)
        rec = recs.setdefault(key, {"metric":metric,"statement":stmt,"n":0,
                                    "acc":{}, "V":0,"X":0,"AB":0})
        if m == list(MODELS)[0]:      # count n / verdicts once (from first model run)
            rec["n"] += 1
            st = r.get("status")
            rec["V" if st=="VERIFIED" else "X" if st=="VIOLATED" else "AB"] += 1
        rec["acc"].setdefault(m, [0,0])
        rec["acc"][m][1] += 1
        rec["acc"][m][0] += int(_correct(r.get("claimed_value"), r.get("gold_answer") or r.get("answer")))

rows=[]
for (metric,stmt), rec in recs.items():
    row={"metric":metric,"statement":stmt,"n":rec["n"],
         "Verified":rec["V"],"Violated":rec["X"],"Abstain":rec["AB"]}
    for m in MODELS:
        cc,tt = rec["acc"].get(m,[0,0]); row[f"acc_{m}"] = 100*cc/tt if tt else np.nan
    rows.append(row)
cats = pd.DataFrame(rows).sort_values(["statement","n"], ascending=[True,False]).reset_index(drop=True)
STMT_COLORS = {"Income stmt":"#1D4ED8","Balance sheet":"#0F766E","Cash flow":"#B91C1C","?":"#6B7280"}
cats


## Question categories

In [ ]:
# Question categories: count by metric, colored by statement
d = cats.sort_values("n")
plt.figure(figsize=(8,5))
plt.barh(d["metric"], d["n"], color=[STMT_COLORS[s] for s in d["statement"]])
for i,(n,) in enumerate(zip(d["n"])): plt.text(n+2, i, str(n), va="center", fontsize=8)
plt.xlabel("# questions"); plt.title(f"Question categories (n={cats['n'].sum()})")
from matplotlib.patches import Patch
plt.legend(handles=[Patch(color=c,label=s) for s,c in STMT_COLORS.items() if s!="?"], loc="lower right", fontsize=8)
plt.tight_layout(); plt.show()


## LLM accuracy by question type and model

The capability spread: haiku is strong, the open 8B models are near the floor.

In [ ]:
# LLM accuracy by metric x model (grouped bars)
d = cats.sort_values(["statement","n"], ascending=[True,False])
mcols = [c for c in cats.columns if c.startswith("acc_")]
x = np.arange(len(d)); w = 0.8/len(mcols)
plt.figure(figsize=(11,5))
for i,c in enumerate(mcols):
    plt.bar(x+i*w, d[c], w, label=c.replace("acc_",""))
plt.xticks(x+w*(len(mcols)-1)/2, d["metric"], rotation=45, ha="right", fontsize=8)
plt.ylabel("LLM answer accuracy (%)"); plt.ylim(0,105)
plt.title("LLM accuracy by question type and model"); plt.legend(); plt.tight_layout(); plt.show()


## LLM accuracy by statement type

Cash-flow questions (multi-item signed sums) are where every model collapses.

In [ ]:
# LLM accuracy by statement type x model
mcols = [c for c in cats.columns if c.startswith("acc_")]
g = cats.groupby("statement").apply(
    lambda t: pd.Series({c.replace("acc_",""): np.average(t[c], weights=t["n"]) for c in mcols}))
g = g.reindex(["Income stmt","Balance sheet","Cash flow"])
g.plot(kind="bar", figsize=(7,4.5)); plt.ylabel("LLM accuracy (%)"); plt.ylim(0,105)
plt.title("LLM accuracy by statement type (the cash-flow cliff)")
plt.xticks(rotation=0); plt.tight_layout(); plt.show()


## VeriFin verdicts by question type

Abstention concentrates on the hardest types; false-accepts are 0 throughout.

## Headline figure — VeriFin is sound (zero false accepts)

One figure, two panels: **(a)** false accepts per verification method on both datasets, **(b)** per answer model. VeriFin = 0 everywhere; every neural baseline and every raw model leaks. Saved to `figures/soundness.{pdf,png}`.

In [ ]:
# Combined soundness figure (two panels). Zero bars get a visible baseline stub +
# green "0" so they read as measured zeros, not missing data.
# Headline counts come from the candidate-specific canonical scorer.
_paper_metrics = pd.read_csv(ROOT/"results/paper_metrics.csv")
def _metric(dataset, group, run):
    row = _paper_metrics[(
        (_paper_metrics["dataset"] == dataset)
        & (_paper_metrics["group"] == group)
        & (_paper_metrics["run"] == run)
    )]
    assert len(row) == 1
    return {"FA": int(row.iloc[0]["FA"])}
def _method_panel(dataset):
    return {
        "raw-LLM": _metric(dataset, "baselines", "direct-llm"),
        "LLM-judge": _metric(dataset, "baselines", "llm-judge"),
        "judge+formula": _metric(dataset, "baselines", "judge-plus-formula"),
        "PoT": _metric(dataset, "baselines", "program-of-thought"),
        "VerifiQA": _metric(dataset, "answer_models", "claude-haiku-4.5"),
    }
res = {"Multicompany (n=600)": _method_panel("xbrlfiling"),
       "FinanceBench (n=67)": _method_panel("financebench")}
def _fa_by_model(path):
    import re
    def nums(t):
        o=[]
        for x in re.findall(r"-?\d[\d,]*\.?\d*",str(t or "")):
            try:o.append(abs(float(x.replace(",",""))))
            except ValueError:pass
        return o
    def ok(cv,g):
        if cv is None:return False
        t=nums(g); return any(abs(abs(float(cv))-x)<=max(1.0,x*0.01) for x in t) if t else False
    raw=vq=0
    for line in Path(path).read_text().splitlines():
        if not line.strip():continue
        r=json.loads(line); cv=r.get("claimed_value"); g=r.get("gold_answer") or r.get("answer")
        good=ok(cv,g)
        if cv is not None and not good: raw+=1
        if r.get("status")=="VERIFIED" and not good: vq+=1
    return raw,vq
def label_bars(ax,bars):
    for r in bars:
        h=r.get_height(); cx=r.get_x()+r.get_width()/2
        if h==0:
            ax.plot([r.get_x()+0.05,r.get_x()+r.get_width()-0.05],[0,0],
                    color=r.get_facecolor(),lw=3.5,solid_capstyle="butt",zorder=5)
            ax.annotate("0",(cx,0),xytext=(0,5),textcoords="offset points",
                        ha="center",fontsize=8.5,color="#2e7d32",fontweight="bold")
        else:
            ax.annotate(str(int(h)),(cx,h),xytext=(0,2),textcoords="offset points",
                        ha="center",fontsize=7.5,color="#333")
with plt.rc_context({"font.family":"serif","font.serif":["Times New Roman","DejaVu Serif"],
                     "font.size":10,"axes.linewidth":0.7}):
    fig,(axL,axR)=plt.subplots(1,2,figsize=(10.2,3.6),gridspec_kw={"width_ratios":[1.35,1]})
    methods=["raw-LLM","LLM-judge","judge+formula","PoT","VerifiQA"]
    mlab={"raw-LLM":"raw-LLM","LLM-judge":"LLM-judge","judge+formula":"judge\n+formula",
          "PoT":"PoT","VerifiQA":"VeriFin\n(ours)"}
    mc=res["Multicompany (n=600)"]; fb=res["FinanceBench (n=67)"]
    mcv=[mc[m]["FA"] for m in methods]; fbv=[fb[m]["FA"] for m in methods]
    x=np.arange(len(methods)); w=0.38
    bA1=axL.bar(x-w/2,mcv,w,color="#4c72b0",label="Multicompany (n=600)")
    bA2=axL.bar(x+w/2,fbv,w,color="#dd8452",label="FinanceBench (n=67)")
    axL.set_xticks(x); axL.set_xticklabels([mlab[m] for m in methods],fontsize=8.5)
    axL.set_ylabel("False accepts  (wrong answers approved)")
    axL.set_title("(a)  By verification method",fontsize=10); axL.legend(frameon=False,fontsize=8)
    names=["claude\n(haiku)","qwen2.5\n-7b","llama3.1\n-8b"]
    dirs=["mc_calc_detsmt_v2_haiku","mc_calc_qwen2.5-7b","mc_calc_llama3.1-8b"]
    raw,vq=[],[]
    for d in dirs:
        a,b=_fa_by_model(ROOT/"results"/d/"results.jsonl"); raw.append(a); vq.append(b)
    xr=np.arange(len(names))
    bB1=axR.bar(xr-w/2,raw,w,color="#c44e52",label="raw-LLM (accept all)")
    bB2=axR.bar(xr+w/2,vq,w,color="#4c72b0",label="VeriFin (ours)")
    axR.set_xticks(xr); axR.set_xticklabels(names,fontsize=8.5)
    axR.set_title("(b)  By answer model  (Multicompany)",fontsize=10); axR.legend(frameon=False,fontsize=8)
    for ax,bars in ((axL,[bA1,bA2]),(axR,[bB1,bB2])):
        for bb in bars: label_bars(ax,bb)
        ax.grid(True,axis="y",lw=0.4,alpha=0.4); ax.set_axisbelow(True)
        ax.spines[["top","right"]].set_visible(False); ax.margins(y=0.12)
    fig.tight_layout()
    fig.savefig(ROOT/"figures/soundness.pdf",bbox_inches="tight")
    fig.savefig(ROOT/"figures/soundness.png",dpi=200,bbox_inches="tight")
    plt.show()


In [ ]:
# Publication figure: model accuracy + aligned VeriFin outcomes by question category
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Patch

family_order = ["Income stmt", "Balance sheet", "Cash flow"]
metric_order = {
    "Income stmt": ["Gross profit", "Operating income", "Operating expenses",
                    "Income before taxes", "Net income"],
    "Balance sheet": ["Current assets", "Non-current assets", "Current liabilities",
                      "Non-current liab.", "Total liabilities",
                      "Shareholders equity", "Net PP&E"],
    "Cash flow": ["Cash: operations", "Cash: investing", "Cash: financing"],
}
order = [metric for family in family_order for metric in metric_order[family]]
plot_df = (cats.set_index("metric").loc[order].reset_index()).copy()

accuracy_columns = ["acc_claude", "acc_qwen2.5-7b", "acc_llama3.1-8b"]
model_labels = ["Claude", "Qwen2.5-7B", "Llama3.1-8B"]
accuracy = plot_df[accuracy_columns].to_numpy(dtype=float)
y = np.arange(len(plot_df))

ink = "#172033"
muted = "#667085"
grid = "#D8DEE9"
verified_color = "#245B8A"
violated_color = "#D59B32"
abstain_color = "#B8C0CC"
heatmap_cmap = LinearSegmentedColormap.from_list(
    "verifiqa_blue", ["#F6F8FB", "#DCE8F2", "#8FB5D1", "#245B8A"]
)

with plt.rc_context({
    "font.family": "DejaVu Sans",
    "font.size": 9,
    "axes.titlesize": 10.5,
    "axes.labelsize": 9,
    "xtick.labelsize": 8.5,
    "ytick.labelsize": 8.5,
}):
    fig = plt.figure(figsize=(11.8, 7.2), facecolor="white")
    gs = fig.add_gridspec(1, 2, width_ratios=[1.05, 1.35], wspace=0.11)
    ax_h = fig.add_subplot(gs[0, 0])
    ax_b = fig.add_subplot(gs[0, 1], sharey=ax_h)

    # (a) Accuracy heatmap
    image = ax_h.imshow(accuracy, cmap=heatmap_cmap, vmin=0, vmax=100,
                        aspect="auto", interpolation="nearest")
    for row in range(accuracy.shape[0]):
        for col in range(accuracy.shape[1]):
            value = accuracy[row, col]
            text_color = "white" if value >= 68 else ink
            ax_h.text(col, row, f"{value:.1f}", ha="center", va="center",
                      color=text_color, fontsize=8.3, fontweight="semibold")
    ax_h.set_xticks(np.arange(len(model_labels)), model_labels)
    ax_h.xaxis.tick_top()
    ax_h.tick_params(axis="x", length=0, pad=7, colors=ink)
    category_labels = [f"{m}  ($n$={n})" for m, n in zip(plot_df["metric"], plot_df["n"])]
    ax_h.set_yticks(y, category_labels)
    ax_h.tick_params(axis="y", length=0, pad=7, colors=ink)
    ax_h.set_title("(a) Answer accuracy (%)", loc="left", pad=34,
                   color=ink, fontweight="semibold")
    for spine in ax_h.spines.values():
        spine.set_visible(False)
    ax_h.set_xticks(np.arange(-0.5, accuracy.shape[1], 1), minor=True)
    ax_h.set_yticks(np.arange(-0.5, accuracy.shape[0], 1), minor=True)
    ax_h.grid(which="minor", color="white", linewidth=1.5)
    ax_h.tick_params(which="minor", bottom=False, left=False)

    # (b) Aligned verdict counts from the Claude run
    verified = plot_df["Verified"].to_numpy()
    violated = plot_df["Violated"].to_numpy()
    abstain = plot_df["Abstain"].to_numpy()
    ax_b.barh(y, verified, color=verified_color, height=0.66, label="Verified")
    ax_b.barh(y, violated, left=verified, color=violated_color, height=0.66, label="Violated")
    ax_b.barh(y, abstain, left=verified + violated, color=abstain_color,
              height=0.66, label="Abstain")
    for row, segments in enumerate(zip(verified, violated, abstain)):
        left = 0
        for value, color_name in zip(segments, ["white", ink, ink]):
            if value >= 5:
                ax_b.text(left + value / 2, row, str(int(value)), ha="center",
                          va="center", fontsize=8, color=color_name,
                          fontweight="semibold")
            left += value
    ax_b.set_xlim(0, max(plot_df["n"]) * 1.02)
    ax_b.set_xlabel("Number of questions", color=muted, labelpad=8)
    ax_b.set_title("(b) VeriFin outcomes (Claude)", loc="left", pad=34,
                   color=ink, fontweight="semibold")
    ax_b.tick_params(axis="y", left=False, labelleft=False)
    ax_b.tick_params(axis="x", colors=muted)
    ax_b.grid(axis="x", color=grid, linewidth=0.6, alpha=0.8)
    ax_b.set_axisbelow(True)
    ax_b.spines[["top", "right", "left"]].set_visible(False)
    ax_b.spines["bottom"].set_color(grid)

    # Family separators and labels shared across both panels.
    family_starts = {}
    cursor = 0
    for family in family_order:
        family_starts[family] = cursor
        cursor += len(metric_order[family])
    boundaries = [len(metric_order["Income stmt"]),
                  len(metric_order["Income stmt"]) + len(metric_order["Balance sheet"])]
    for boundary in boundaries:
        for ax in (ax_h, ax_b):
            ax.axhline(boundary - 0.5, color=ink, linewidth=1.1, alpha=0.6)
    family_display = {"Income stmt": "INCOME STATEMENT",
                      "Balance sheet": "BALANCE SHEET",
                      "Cash flow": "CASH FLOW"}
    for family, start in family_starts.items():
        midpoint = start + (len(metric_order[family]) - 1) / 2
        ax_h.text(-0.72, midpoint, family_display[family],
                  transform=ax_h.get_yaxis_transform(), rotation=90,
                  ha="center", va="center", fontsize=7.5, color=muted,
                  fontweight="bold", clip_on=False)

    legend_handles = [Patch(facecolor=verified_color, label="Verified"),
                      Patch(facecolor=violated_color, label="Violated"),
                      Patch(facecolor=abstain_color, label="Abstain")]
    ax_b.legend(handles=legend_handles, ncol=3, frameon=False, loc="upper left",
                bbox_to_anchor=(0, 1.055), borderaxespad=0, fontsize=8.5,
                handlelength=1.2, columnspacing=1.3)

    fig.subplots_adjust(left=0.28, right=0.985, top=0.93, bottom=0.10)

    output_pdf = ROOT / "figures/category_model_performance.pdf"
    output_png = ROOT / "figures/category_model_performance.png"
    fig.savefig(output_pdf, bbox_inches="tight")
    fig.savefig(output_png, dpi=240, bbox_inches="tight", facecolor="white")
    plt.show()

print(f"Saved {output_pdf}")
print(f"Saved {output_png}")


In [ ]:

# Main-paper figure: upright XBRLBench rejection bars only.
# Optional repair side-channel outputs are ignored so deterministic SMT stays fixed.
import json
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import Patch

paper_model_runs = [
    ("Claude Haiku 4.5", ROOT / "results/mc_calc_detsmt_v2_haiku/results.jsonl"),
    ("Fino1", ROOT / "results/latest/mc-fino-30b-latest/results.jsonl"),
    ("Qwen3-30B", ROOT / "results/latest/mc-qwen-30b-latest/results.jsonl"),
    ("Qwen2.5-7B", ROOT / "results/mc_calc_detsmt_v2_qwen2.5-7b/results.jsonl"),
    ("Llama-3.1-8B", ROOT / "results/mc_calc_detsmt_v2_llama3.1-8b/results.jsonl"),
]
MODEL_ORDER = [model for model, _ in paper_model_runs]

xbrl_question_by_id = {
    row["financebench_id"]: row
    for row in (
        json.loads(line)
        for line in (
            ROOT / "data/multicompany_provable/data/provable_calcrequired.jsonl"
        ).read_text().splitlines()
        if line.strip()
    )
}

FAMILY_TERMS = [
    ("gross profit", "Income statement"),
    ("operating income", "Income statement"),
    ("operating expenses", "Income statement"),
    ("income before income taxes", "Income statement"),
    ("net income", "Income statement"),
    ("total current assets", "Balance sheet"),
    ("total non-current assets", "Balance sheet"),
    ("total current liabilities", "Balance sheet"),
    ("total non-current liabilities", "Balance sheet"),
    ("total liabilities", "Balance sheet"),
    ("total shareholders", "Balance sheet"),
    ("net property", "Balance sheet"),
    ("net cash from operating", "Cash flow"),
    ("net cash from investing", "Cash flow"),
    ("net cash from financing", "Cash flow"),
]
FAMILY_ORDER = ["Income statement", "Balance sheet", "Cash flow"]


def _statement_family(question):
    lowered = question.lower()
    for term, family in FAMILY_TERMS:
        if term in lowered:
            return family
    raise ValueError(f"Unmapped XBRLBench question: {question}")


def _paper_numbers(text):
    values = []
    for token in re.findall(r"-?\d[\d,]*\.?\d*", str(text or "")):
        try:
            values.append(abs(float(token.replace(",", ""))))
        except ValueError:
            pass
    return values


def _paper_correct(claimed_value, gold):
    """Paper convention: magnitude match within max($1m, 1%)."""
    if claimed_value is None:
        return False
    gold_values = _paper_numbers(gold)
    return any(
        abs(abs(float(claimed_value)) - value) <= max(1.0, value * 0.01)
        for value in gold_values
    )


def _xbrl_caught_by_family(path):
    caught = {family: 0 for family in FAMILY_ORDER}
    incorrect_numeric = {family: 0 for family in FAMILY_ORDER}
    false_accepts = 0
    unresolved_incorrect = 0
    rows = [
        json.loads(line)
        for line in Path(path).read_text().splitlines()
        if line.strip()
    ]
    assert len(rows) == 600
    assert len({row["id"] for row in rows}) == 600
    assert all(row["id"] in xbrl_question_by_id for row in rows)
    for row in rows:
        claimed_value = row.get("claimed_value")
        if claimed_value is None or _paper_correct(
            claimed_value,
            row.get("gold_answer") or row.get("answer"),
        ):
            continue
        family = _statement_family(xbrl_question_by_id[row["id"]]["question"])
        incorrect_numeric[family] += 1
        if row.get("status") == "VIOLATED":
            caught[family] += 1
        elif row.get("status") == "VERIFIED":
            false_accepts += 1
        else:
            unresolved_incorrect += 1
    return caught, incorrect_numeric, false_accepts, unresolved_incorrect


xbrl_caught_rows = []
xbrl_incorrect_rows = []
xbrl_false_accepts = {}
xbrl_unresolved = {}
for model, path in paper_model_runs:
    caught, incorrect, false_accepts, unresolved = _xbrl_caught_by_family(path)
    xbrl_caught_rows.append({"Answer model": model, **caught})
    xbrl_incorrect_rows.append({"Answer model": model, **incorrect})
    xbrl_false_accepts[model] = false_accepts
    xbrl_unresolved[model] = unresolved

xbrl_caught = pd.DataFrame(xbrl_caught_rows).set_index("Answer model")
xbrl_incorrect = pd.DataFrame(xbrl_incorrect_rows).set_index("Answer model")
assert xbrl_caught[FAMILY_ORDER].equals(xbrl_incorrect[FAMILY_ORDER])
assert all(value == 0 for value in xbrl_false_accepts.values())
assert all(value == 0 for value in xbrl_unresolved.values())
assert int(xbrl_caught[FAMILY_ORDER].to_numpy().sum()) == 1372

MODEL_TICK_LABELS = [
    "Claude\nHaiku 4.5",
    "Fino1",
    "Qwen3\n30B",
    "Qwen2.5\n7B",
    "Llama3.1\n8B",
]
XBRL_SPECS = [
    ("Income statement", "Income statement", "#28618F", "#1D496C", ""),
    ("Balance sheet", "Balance sheet", "#2A8C82", "#1C625B", "///"),
    ("Cash flow", "Cash flow", "#D89B2B", "#8C631B", "..."),
]
INK, MUTED, GRID = "#17212B", "#5B6573", "#D4DAE1"

with plt.rc_context(
    {
        "font.family": "serif",
        "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
        "font.size": 7.5,
        "axes.linewidth": 0.65,
        "hatch.linewidth": 0.45,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
    }
):
    fig, ax = plt.subplots(figsize=(5.60, 3.05))
    ordered = xbrl_caught.reindex(MODEL_ORDER)
    model_x = np.arange(len(MODEL_ORDER)) * 1.15
    bar_width = 0.33
    offsets = (-0.35, 0.0, 0.35)

    for (column, _, facecolor, edgecolor, hatch), offset in zip(
        XBRL_SPECS,
        offsets,
    ):
        values = ordered[column].to_numpy()
        bars = ax.bar(
            model_x + offset,
            values,
            width=bar_width,
            color=facecolor,
            edgecolor=edgecolor,
            linewidth=0.55,
            hatch=hatch,
            zorder=3,
        )
        for bar, value in zip(bars, values):
            ax.annotate(
                f"{int(value)}",
                (bar.get_x() + bar.get_width() / 2, value),
                xytext=(0, 3),
                textcoords="offset points",
                ha="center",
                va="bottom",
                fontsize=6.6,
                color=INK,
                fontweight="bold",
                zorder=4,
            )

    ax.set_title(
        "XBRLBench ($n=600$/model)",
        loc="left",
        fontsize=8.4,
        fontweight="bold",
        color=INK,
        pad=28,
    )
    ax.set_xlim(model_x[0] - 0.60, model_x[-1] + 0.60)
    ax.set_ylim(0, 265)
    ax.set_yticks([0, 50, 100, 150, 200, 250])
    ax.set_ylabel(
        "Incorrect answers caught by VeriFin (count)",
        fontsize=6.6,
        color=INK,
        labelpad=5,
    )
    ax.set_xticks(model_x, MODEL_TICK_LABELS, fontsize=6.9, color=INK)
    ax.grid(axis="y", color=GRID, linewidth=0.45, alpha=0.9, zorder=1)
    ax.tick_params(axis="x", length=0, pad=5)
    ax.tick_params(
        axis="y",
        length=2.3,
        width=0.55,
        colors=MUTED,
        labelsize=6.6,
    )
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.spines["bottom"].set_color("#AEB5BE")

    legend_handles = [
        Patch(
            facecolor=facecolor,
            edgecolor=edgecolor,
            linewidth=0.55,
            hatch=hatch,
            label=label,
        )
        for _, label, facecolor, edgecolor, hatch in XBRL_SPECS
    ]
    ax.legend(
        handles=legend_handles,
        loc="lower left",
        bbox_to_anchor=(0.0, 1.01),
        ncol=3,
        frameon=False,
        fontsize=6.6,
        handlelength=1.15,
        handletextpad=0.4,
        columnspacing=0.8,
        borderaxespad=0,
    )

    fig.subplots_adjust(left=0.075, right=0.995, top=0.72, bottom=0.22)

    output_pdf = ROOT / "figures/category_model_accuracy_fixed_verifier.pdf"
    output_png = ROOT / "figures/category_model_accuracy_fixed_verifier.png"
    fig.savefig(output_pdf, bbox_inches="tight", pad_inches=0.02, facecolor="white")
    fig.savefig(
        output_png,
        dpi=300,
        bbox_inches="tight",
        pad_inches=0.02,
        facecolor="white",
    )
    plt.show()
